In [ ]:
# Clean the daily source table with src_jup validation and normalization helpers.
# —— 日表清洗：字段校验 → 类型规范化 → A股与样本期过滤 → 去重 → 有效交易 ——
required_daily_columns = [
    stock_code, date, close_price, total_shares, daily_ret, daily_rfret,
    trade_volumn, trade_amount, eps, pb, roe, public_date, end_date,
]
require_columns(df, required_daily_columns, '日股表')

df[stock_code] = normalize_stock_code(df[stock_code])
df[date] = normalize_datetime(df[date])
for column in [
    close_price, total_shares, daily_ret, daily_rfret, trade_volumn,
    trade_amount, eps, pb, roe,
]:
    df[column] = normalize_numeric(df[column])

is_a_share = df[stock_code].str[:2].isin(['00', '30', '60'])
is_in_period = df[date].between(start_date, final_date)
df = df.loc[is_a_share & is_in_period].copy()

# (股票, 交易日) 必须唯一：同键重复保留先出现的一条（源文件行序固定）。
df = df.drop_duplicates([stock_code, date], keep='first')
df = df.sort_values([stock_code, date]).reset_index(drop=True)

# 有效交易：收盘价、总股本、成交量均大于零，且当日收益可计算。
valid_trade = (
    df[close_price].gt(0)
    & df[total_shares].gt(0)
    & df[trade_volumn].gt(0)
    & df[daily_ret].notna()
)
df = df.loc[valid_trade].reset_index(drop=True)



In [ ]:
# Clean the local monthly table and create return fields.
# —— 月表清洗：类型规范化 → A股与样本期过滤 → (股票, 月) 唯一 ——
monthly[stock_code] = normalize_stock_code(monthly[stock_code])
monthly[date] = normalize_datetime(monthly[date])
monthly_is_a_share = monthly[stock_code].str[:2].isin(['00', '30', '60'])
monthly_is_in_period = monthly[date].between(start_date, final_date)
monthly = monthly.loc[monthly_is_a_share & monthly_is_in_period].copy()
monthly = monthly.sort_values([stock_code, date])
monthly = monthly.drop_duplicates([stock_code, date], keep='first')

# 同一自然月只保留先出现的一条，让 (股票, 月) 成为唯一主键。
monthly[ym] = monthly[date].dt.to_period('M')
monthly = monthly.drop_duplicates([stock_code, ym], keep='first')
monthly = monthly.reset_index(drop=True)

# 月收益与月超额收益：Monret 为月收益率，Monrfret 为月度无风险利率。
monthly[monthly_return_raw] = normalize_numeric(monthly[monthly_return_raw])
monthly[monthly_rfreturn_raw] = normalize_numeric(monthly[monthly_rfreturn_raw])
monthly[monthly_return] = monthly[monthly_return_raw]
monthly[monthly_excess_ret] = monthly[monthly_return] - monthly[monthly_rfreturn_raw]



In [ ]:
# Clean each financial statement with the existing src_jup functions.
# —— 财报表清洗：字段契约、类型规范化与去重由 src_jup 清洗函数执行 ——
cf_df = cf_cleaning(cf_df)
bs_df = bs_cleaning(bs_df)
fas_df = fas_cleaning(fas_df)

